# Long Document Processor

LLMs have finite context windows. When a document exceeds that limit, you cannot simply pass it in whole.
This notebook covers the core techniques for handling long documents:

1. **Chunking Strategies** — how to split text into model-sized pieces
2. **Map-Reduce Processing** — apply a function to every chunk, then aggregate
3. **Hierarchical Summarization** — build a tree of summaries bottom-up
4. **Position-Aware Assembly (Sandwich Pattern)** — place the most important content where the model pays attention

All examples use mock LLM calls so the notebook runs without API keys.

In [ ]:
# Cell 2 — Imports and token counting helper
import re
import textwrap
from typing import Callable

try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text: str) -> int:
        """Count tokens using tiktoken cl100k_base (GPT-4 / Claude compatible)."""
        return len(_enc.encode(text))
except ImportError:
    # Fallback: rough approximation (1 token ~= 4 chars)
    def count_tokens(text: str) -> int:
        """Approximate token count when tiktoken is unavailable."""
        return max(1, len(text) // 4)
    print("tiktoken not installed — using character-based approximation.")

print(f"count_tokens('Hello, world!') = {count_tokens('Hello, world!')}")

## 1. Chunking Strategies

Chunking is the first step in any long-document pipeline. The right strategy depends on the document structure and downstream task.

| Strategy | Best for | Key trade-off |
| :--- | :--- | :--- |
| Fixed-size | Uniform text, fast pipelines | May cut mid-sentence |
| Semantic | Structured prose (paragraphs) | Chunk sizes vary |
| Sliding window | Retrieval / QA (need overlap) | More chunks, more cost |

In [ ]:
# Cell 4 — Three chunking functions

def fixed_size_chunk(text: str, chunk_size: int = 512, overlap: int = 64) -> list[str]:
    """
    Split text into fixed-size token chunks with optional overlap.

    Args:
        text:       Input text to split.
        chunk_size: Maximum tokens per chunk.
        overlap:    Number of tokens to repeat at the start of each new chunk.

    Returns:
        List of text chunks.
    """
    words = text.split()
    # Approximate: treat each word as ~1.3 tokens
    words_per_chunk = max(1, int(chunk_size / 1.3))
    words_overlap   = max(0, int(overlap / 1.3))
    stride = words_per_chunk - words_overlap

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + words_per_chunk, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += stride
    return chunks


def semantic_chunk(text: str, max_tokens: int = 512) -> list[str]:
    """
    Split text on paragraph boundaries (\\n\\n), merging small paragraphs
    and splitting oversized ones.

    Args:
        text:       Input text.
        max_tokens: Token budget per chunk.

    Returns:
        List of semantically coherent chunks.
    """
    paragraphs = [p.strip() for p in re.split(r"\n\n+", text) if p.strip()]
    chunks: list[str] = []
    current_parts: list[str] = []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = count_tokens(para)

        if para_tokens > max_tokens:
            # Flush current buffer first
            if current_parts:
                chunks.append("\n\n".join(current_parts))
                current_parts, current_tokens = [], 0
            # Split the oversized paragraph by sentences
            sentences = re.split(r"(?<=[.!?])\s+", para)
            sub_parts: list[str] = []
            sub_tokens = 0
            for sent in sentences:
                st = count_tokens(sent)
                if sub_tokens + st > max_tokens and sub_parts:
                    chunks.append(" ".join(sub_parts))
                    sub_parts, sub_tokens = [], 0
                sub_parts.append(sent)
                sub_tokens += st
            if sub_parts:
                chunks.append(" ".join(sub_parts))
        elif current_tokens + para_tokens > max_tokens:
            # Flush and start a new chunk
            if current_parts:
                chunks.append("\n\n".join(current_parts))
            current_parts = [para]
            current_tokens = para_tokens
        else:
            current_parts.append(para)
            current_tokens += para_tokens

    if current_parts:
        chunks.append("\n\n".join(current_parts))
    return chunks


def sliding_window_chunk(text: str, window_size: int = 512, stride: int = 384) -> list[str]:
    """
    Sliding window chunking: each chunk overlaps with the previous by
    (window_size - stride) tokens. Useful for retrieval tasks.

    Args:
        text:        Input text.
        window_size: Tokens per window.
        stride:      Tokens to advance between windows.

    Returns:
        List of overlapping text chunks.
    """
    words = text.split()
    words_per_window = max(1, int(window_size / 1.3))
    words_stride     = max(1, int(stride / 1.3))

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + words_per_window, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += words_stride
    return chunks


print("Chunking functions defined: fixed_size_chunk, semantic_chunk, sliding_window_chunk")

In [ ]:
# Cell 5 — Demo with sample text

# Build a ~2000-word sample document by repeating a paragraph 20 times
_base_paragraph = (
    "Context engineering is the discipline of designing and managing the information "
    "provided to a large language model within its context window. Unlike prompt engineering, "
    "which focuses on the wording of individual instructions, context engineering considers "
    "the full assembly pipeline: what to include, how to compress it, where to place it, "
    "and how to adapt it dynamically as a conversation evolves. "
    "Effective context engineering directly impacts model accuracy, cost, and latency."
)
sample_text = "\n\n".join([_base_paragraph] * 20)

total_tokens = count_tokens(sample_text)
print(f"Sample text: {len(sample_text):,} chars | {total_tokens:,} tokens\n")

# Run all three strategies
strategies = {
    "fixed_size (512t, 64 overlap)": fixed_size_chunk(sample_text, chunk_size=512, overlap=64),
    "semantic   (max 512t)        ": semantic_chunk(sample_text, max_tokens=512),
    "sliding_window (512t, s=384) ": sliding_window_chunk(sample_text, window_size=512, stride=384),
}

print(f"{'Strategy':<40} {'Chunks':>6}  {'Avg tokens':>10}  {'Min':>6}  {'Max':>6}")
print("-" * 72)
for name, chunks in strategies.items():
    token_counts = [count_tokens(c) for c in chunks]
    avg = sum(token_counts) / len(token_counts)
    print(f"{name:<40} {len(chunks):>6}  {avg:>10.1f}  {min(token_counts):>6}  {max(token_counts):>6}")

## 2. Map-Reduce Processing

Map-Reduce is the standard pattern for processing documents that exceed the context window:

1. **Map** — apply a function (e.g., summarize, extract) to each chunk independently
2. **Reduce** — aggregate the per-chunk results into a single output

The map step is embarrassingly parallel. The reduce step may itself need to be recursive if the combined map outputs are still too large.

In [ ]:
# Cell 7 — MapReduceProcessor class

class MapReduceProcessor:
    """
    Generic map-reduce processor for long documents.

    Args:
        map_fn:    Callable[[str], str] — applied to each chunk.
        reduce_fn: Callable[[list[str]], str] — aggregates map outputs.
    """

    def __init__(self, map_fn: Callable[[str], str], reduce_fn: Callable[[list[str]], str]):
        self.map_fn    = map_fn
        self.reduce_fn = reduce_fn

    def process(self, text: str, chunk_size: int = 512) -> str:
        """
        Split text into chunks, map over each, then reduce.

        Args:
            text:       Full document text.
            chunk_size: Token budget per chunk (uses fixed_size_chunk).

        Returns:
            Final aggregated result string.
        """
        chunks = fixed_size_chunk(text, chunk_size=chunk_size, overlap=0)
        print(f"  [map]    Processing {len(chunks)} chunks...")
        mapped = [self.map_fn(chunk) for chunk in chunks]
        print(f"  [reduce] Aggregating {len(mapped)} results...")
        return self.reduce_fn(mapped)


# --- Mock functions (no real LLM required) ---

def mock_summarize(chunk: str) -> str:
    """Mock map function: return first sentence of the chunk."""
    first_sentence = chunk.split(".")[0].strip()
    return first_sentence + "."


def mock_aggregate(summaries: list[str]) -> str:
    """Mock reduce function: join all summaries with a newline."""
    return "\n".join(f"- {s}" for s in summaries)


# Demo
processor = MapReduceProcessor(map_fn=mock_summarize, reduce_fn=mock_aggregate)
print("Running MapReduceProcessor on sample_text...\n")
result = processor.process(sample_text, chunk_size=512)

print(f"\nFinal result ({count_tokens(result)} tokens):")
print(textwrap.indent(result[:400] + ("..." if len(result) > 400 else ""), "  "))

## 3. Hierarchical Summarization

When the map outputs are themselves too large to reduce in one pass, build a **summary tree**:

- **Level 0** — original chunks
- **Level 1** — summaries of groups of Level-0 chunks
- **Level 2** — summaries of groups of Level-1 summaries
- ...
- **Root** — final single summary

This is equivalent to a B-tree merge, applied to text.

In [ ]:
# Cell 9 — HierarchicalSummarizer

class HierarchicalSummarizer:
    """
    Build a summary tree over a list of text chunks.

    Args:
        summarize_fn:  Callable[[str], str] — summarizes a single text.
        branch_factor: How many chunks to merge per level (default 4).
    """

    def __init__(self, summarize_fn: Callable[[str], str], branch_factor: int = 4):
        self.summarize_fn  = summarize_fn
        self.branch_factor = branch_factor

    def build_tree(self, chunks: list[str]) -> dict:
        """
        Recursively build a summary tree.

        Returns:
            dict with keys:
              'levels': list of lists (level 0 = original chunks)
              'root':   final summary string
        """
        levels = [chunks[:]]
        current = chunks[:]

        while len(current) > 1:
            next_level = []
            for i in range(0, len(current), self.branch_factor):
                group    = current[i : i + self.branch_factor]
                combined = "\n\n".join(group)
                summary  = self.summarize_fn(combined)
                next_level.append(summary)
            levels.append(next_level)
            current = next_level

        return {"levels": levels, "root": current[0]}


# Mock summarize: first 80 characters
def mock_summarize_short(text: str) -> str:
    return text[:80].rstrip() + "..."


# Build tree over semantic chunks of sample_text
sem_chunks = semantic_chunk(sample_text, max_tokens=256)
summarizer = HierarchicalSummarizer(summarize_fn=mock_summarize_short, branch_factor=4)
tree = summarizer.build_tree(sem_chunks)

print(f"Input chunks: {len(sem_chunks)}")
for lvl_idx, level in enumerate(tree["levels"]):
    avg_tok = sum(count_tokens(n) for n in level) / len(level)
    label = "(original)" if lvl_idx == 0 else "(summaries)"
    print(f"  Level {lvl_idx} {label}: {len(level):>3} nodes | avg {avg_tok:.0f} tokens each")

print(f"\nRoot summary ({count_tokens(tree['root'])} tokens):")
print(f"  {tree['root']}")

## 4. Position-Aware Assembly (Sandwich Pattern)

Research shows LLMs pay most attention to content at the **beginning** and **end** of the context window — the "Lost in the Middle" effect (Liu et al., 2024).

The **Sandwich Pattern** exploits this:

```
[Task instruction]          <- start (high attention)
[Background / static docs]
[Less relevant chunks]
[Most relevant chunk]       <- just before query (high attention)
[User query]
[Task instruction repeated] <- end (high attention)
```

This ensures critical instructions and the most relevant evidence are never "lost in the middle".

In [ ]:
# Cell 11 — position_aware_assemble

def position_aware_assemble(
    task_instruction: str,
    background: str,
    relevant_chunks: list[str],
    query: str,
) -> tuple[str, int]:
    """
    Assemble a context string using the Sandwich Pattern.

    Layout:
        1. task_instruction          (start — high attention)
        2. background                (static, cacheable)
        3. relevant_chunks[1:]       (less relevant chunks in the middle)
        4. relevant_chunks[0]        (most relevant chunk just before query)
        5. query
        6. task_instruction repeated (end — high attention)

    Args:
        task_instruction: The instruction / system prompt for the task.
        background:       Static background knowledge.
        relevant_chunks:  Retrieved chunks, ordered most-relevant first.
        query:            The user's question.

    Returns:
        Tuple of (assembled_context: str, total_tokens: int).
    """
    separator = "\n" + "-" * 60 + "\n"
    parts = []

    # 1. Instruction at the top
    parts.append(f"[INSTRUCTION]\n{task_instruction}")

    # 2. Background
    if background.strip():
        parts.append(f"[BACKGROUND]\n{background}")

    # 3. Less-relevant chunks in the middle (indices 1+)
    for i, chunk in enumerate(relevant_chunks[1:], start=2):
        parts.append(f"[CONTEXT CHUNK {i}]\n{chunk}")

    # 4. Most relevant chunk just before the query
    if relevant_chunks:
        parts.append(f"[MOST RELEVANT CHUNK]\n{relevant_chunks[0]}")

    # 5. Query
    parts.append(f"[QUERY]\n{query}")

    # 6. Instruction repeated at the end
    parts.append(f"[REMINDER]\n{task_instruction}")

    assembled = separator.join(parts)
    total_tokens = count_tokens(assembled)
    return assembled, total_tokens


print("position_aware_assemble defined.")

In [ ]:
# Cell 12 — Final demo: all techniques together

print("=" * 70)
print("LONG DOCUMENT PROCESSING — END-TO-END DEMO")
print("=" * 70)

# --- Step 1: Chunk the document ---
print("\n[Step 1] Chunking with sliding_window_chunk (window=512, stride=384)")
chunks = sliding_window_chunk(sample_text, window_size=512, stride=384)
print(f"  Produced {len(chunks)} chunks")

# --- Step 2: Map-Reduce to get a document summary ---
print("\n[Step 2] Map-Reduce summarization")
mr = MapReduceProcessor(map_fn=mock_summarize, reduce_fn=mock_aggregate)
doc_summary = mr.process(sample_text, chunk_size=512)
print(f"  Summary: {count_tokens(doc_summary)} tokens")

# --- Step 3: Hierarchical summarization for a compact root ---
print("\n[Step 3] Hierarchical summarization")
hier = HierarchicalSummarizer(summarize_fn=mock_summarize_short, branch_factor=4)
tree = hier.build_tree(chunks)
root_summary = tree["root"]
print(f"  Tree depth: {len(tree['levels'])} levels")
print(f"  Root summary: {count_tokens(root_summary)} tokens")

# --- Step 4: Sandwich assembly for the final prompt ---
print("\n[Step 4] Position-aware assembly (Sandwich Pattern)")
task_instruction = (
    "You are a research assistant. Answer the user's question using ONLY "
    "the provided context. If the answer is not in the context, say so."
)
query = "What is context engineering and why does it matter?"

# Use first 3 chunks as retrieved results (most relevant first)
top_chunks = chunks[:3]

assembled, total_tokens = position_aware_assemble(
    task_instruction=task_instruction,
    background=root_summary,
    relevant_chunks=top_chunks,
    query=query,
)

print(f"  Assembled context: {total_tokens} tokens")
print(f"  Layout preview (first 600 chars):")
print(textwrap.indent(assembled[:600] + "...", "    "))

# --- Summary table ---
print("\n" + "=" * 70)
print("TOKEN BUDGET SUMMARY")
print("=" * 70)
print(f"  Original document:      {count_tokens(sample_text):>6} tokens")
print(f"  Map-Reduce summary:     {count_tokens(doc_summary):>6} tokens")
print(f"  Hierarchical root:      {count_tokens(root_summary):>6} tokens")
print(f"  Final assembled prompt: {total_tokens:>6} tokens")
compression = (1 - total_tokens / count_tokens(sample_text)) * 100
print(f"  Compression ratio:      {compression:.1f}% reduction")